# Face Image Generation

Generate synthetic facial portraits from CSV prompts using `black-forest-labs/FLUX.2-klein-4B`. Each generated image is saved with a CSV metadata record containing its prompt, random seed, and skin-condition label.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import torch
from diffusers import Flux2KleinPipeline, FlowMatchEulerDiscreteScheduler
from tqdm.auto import tqdm

# Paths (relative to the notebook working directory, or use absolute paths)
PROMPTS_DIR = Path("prompts_files")
OUTPUT_DIR = Path("generated_images")
PROMPT_FILE_PATTERN = "face_generation*.csv"

# Generation settings
MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"
SEED = 42
IMAGE_HEIGHT = 1024
IMAGE_WIDTH = 1024
GUIDANCE_SCALE = 1.0
NUM_INFERENCE_STEPS = 6
MAX_SEQUENCE_LENGTH = 512
SCHEDULER_SHIFT = 3.0
MODEL_DTYPE = torch.bfloat16

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA-enabled GPU is required for this notebook.")

if not PROMPTS_DIR.is_dir():
    raise FileNotFoundError(f"Prompt directory not found: {PROMPTS_DIR}")

prompt_files = sorted(PROMPTS_DIR.glob(PROMPT_FILE_PATTERN))
if not prompt_files:
    raise FileNotFoundError(
        f"No files matching {PROMPT_FILE_PATTERN!r} found in {PROMPTS_DIR}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

pipe = Flux2KleinPipeline.from_pretrained(MODEL_ID, torch_dtype=MODEL_DTYPE)
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(
    pipe.scheduler.config, shift=SCHEDULER_SHIFT
)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)

In [ ]:
for prompt_file in prompt_files:
    df = pd.read_csv(prompt_file)
    if "prompt" not in df.columns:
        raise ValueError(f"Missing 'prompt' column in {prompt_file}")

    group_name = prompt_file.stem
    image_dir = OUTPUT_DIR / group_name
    image_dir.mkdir(parents=True, exist_ok=True)
    metadata = []

    for index, row in tqdm(df.iterrows(), total=len(df), desc=group_name):
        prompt = row["prompt"]
        if pd.isna(prompt) or not str(prompt).strip():
            raise ValueError(f"Empty prompt at row {index + 1} in {prompt_file}")

        skin_value = row.get("skin_condition", "")
        skin_condition = "" if pd.isna(skin_value) else str(skin_value)
        safe_skin = re.sub(r"[^A-Za-z0-9_-]+", "_", skin_condition).strip("_")
        current_seed = SEED + index
        generator = torch.Generator(device="cuda").manual_seed(current_seed)

        image = pipe(
            prompt=str(prompt),
            height=IMAGE_HEIGHT,
            width=IMAGE_WIDTH,
            guidance_scale=GUIDANCE_SCALE,
            num_inference_steps=NUM_INFERENCE_STEPS,
            generator=generator,
            max_sequence_length=MAX_SEQUENCE_LENGTH,
        ).images[0]

        file_name = f"{safe_skin}_{index + 1}.png" if safe_skin else f"{index + 1}.png"
        image_path = image_dir / file_name
        image.save(image_path)

        metadata.append({
            "index": index + 1,
            "seed": current_seed,
            "prompt": str(prompt),
            "skin_condition": skin_condition,
            "file_name": file_name,
            "gender": group_name,
            "image_path": str(image_path),
        })

    pd.DataFrame(metadata, columns=[
        "index", "seed", "prompt", "skin_condition", "file_name", "gender", "image_path"
    ]).to_csv(image_dir / f"{group_name}_metadata.csv", index=False)